# Chains in LangChain

## Outline

- LLMChain
- Sequential Chains
  - SimpleSequentialChain
  - SequentialChain
- Router Chain


In [ ]:
# !pip install langchain langchain-fireworks pandas python-dotenv langchain_community

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import os

from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())  # read local .env file

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.


In [3]:
import pandas as pd

df = pd.read_csv("Data.csv")

In [4]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## Simple Chain


In [11]:
from langchain.chains import SimpleSequentialChain
from langchain_fireworks import ChatFireworks
from langchain.prompts import ChatPromptTemplate
from langchain.chains.llm import LLMChain

In [12]:
llm_model = "accounts/fireworks/models/llama-v3p1-70b-instruct"
llm = ChatFireworks(temperature=0, model=llm_model)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following \
    company:{company_name}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

overall_simple_chain = SimpleSequentialChain(
    chains=[chain_one, chain_two], verbose=False
)

C:\Users\phi.nguyen\AppData\Local\Temp\ipykernel_3908\2567561620.py:11: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain_one = LLMChain(llm=llm, prompt=first_prompt)


In [14]:
product = "Mouse"

In [15]:
print(overall_simple_chain.run(product))

C:\Users\phi.nguyen\AppData\Local\Temp\ipykernel_3908\4066057306.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  print(overall_simple_chain.run(product))


Here are 20-word descriptions for each of the company name suggestions:

1. **MouseMaster**: "Expertly crafted mice for precision and performance, trusted by professionals and gamers alike."
2. **ClickTech**: "Innovative mouse technology for seamless navigation and effortless control."
3. **Rodentia**: "Sleek, modern mice designed for comfort and precision, inspired by nature."
4. **MouseWorks**: "Reliable, functional mice built for everyday use, designed with users in mind."
5. **CursorCraft**: "Handcrafted mice with attention to detail, precision, and a passion for innovation."
6. **PointPro**: "Accurate, precise mice for professionals and gamers who demand the best."
7. **MouseMakers**: "Mice made with care, designed for comfort, and built to last."
8. **GlideTech**: "Smooth, effortless mouse movement for a seamless user experience."
9. **ClickCraft Co.**: "Expertly crafted mice with a focus on quality, precision, and attention to detail."
10. **MouseForge**: "Forging high-quality m

New API


In [ ]:
product = ["Games", "Shoes"]

In [18]:
from langchain.schema.output_parser import StrOutputParser

overall_simple_chain = (
    first_prompt | llm | StrOutputParser() | second_prompt | llm | StrOutputParser()
)
print(overall_simple_chain.batch(product)[1])

Here is a 20-word description for a company that makes... let's say, eco-friendly cleaning products:

"GreenClean: Innovative, sustainable cleaning solutions for a healthier home and planet, naturally effective and gentle on the environment always."


## RunablePassThrough, RunableParallel, RunableLambda


In [19]:
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableLambda,
    RunnableParallel,
)

In [21]:
chain = RunnablePassthrough() | RunnablePassthrough() | RunnablePassthrough()
chain.invoke("hello")

'hello'

In [22]:
def input_to_upper(input: str):
    output = input.upper()
    return output

In [23]:
chain = RunnablePassthrough() | RunnableLambda(input_to_upper) | RunnablePassthrough()
chain.invoke("hello")

'HELLO'

In [ ]:
chain = RunnableParallel({"x": RunnablePassthrough(), "y": RunnablePassthrough()})

In [ ]:
chain.invoke("hello")

In [ ]:
RunnableParallel({"query": "input1", "context": "input2"})

In [24]:
chain = RunnableParallel({"x": RunnablePassthrough(), "y": lambda z: z["input2"]}

In [25]:
chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': {'input': 'hello', 'input2': 'goodbye'}, 'y': 'goodbye'}

### Nested chains - now it gets more complicated!


In [26]:
def find_keys_to_uppercase(input: dict):
    output = input.get("input", "not found").upper()
    return output

In [30]:
chain = RunnableParallel({
    "x": RunnableLambda(find_keys_to_uppercase),
    "y": lambda z: z["input2"],
})

In [31]:
chain.invoke({"input": "hello", "input2": "goodbye"})

{'x': 'HELLO', 'y': 'goodbye'}

## A more complicated chain


Deprecated API


In [34]:
# Deprecated API
from langchain.chains import SequentialChain

llm = ChatFireworks(temperature=0, model=llm_model)

# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:" "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = LLMChain(llm=llm, prompt=first_prompt, output_key="English_Review")

second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:" "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = LLMChain(llm=llm, prompt=second_prompt, output_key="summary")


# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt, output_key="language")

# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(llm=llm, prompt=fourth_prompt, output_key="followup_message")

# overall_chain: input= Review
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary", "followup_message"],
    verbose=True,
)

In [35]:
review = df.Review[5]
results_old = overall_chain(review)
results_old

C:\Users\phi.nguyen\AppData\Local\Temp\ipykernel_3908\562686162.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results_old = overall_chain(review)




> Entering new SequentialChain chain...

> Finished chain.


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'English_Review': 'Here is the translation of the review to English:\n\n"I find the taste mediocre. The foam doesn\'t hold, it\'s weird. I buy the same ones in stores and the taste is much better... Old batch or counterfeit!?"',
 'summary': "The reviewer was disappointed with the taste and texture of the product, suspecting that it may be an old or counterfeit batch since it differed significantly from the same product they've purchased in stores.",
 'followup_message': 'Here is a potential follow-up response in French:\n\n"Je suis désolé d\'entendre que vous n\'avez pas été satisfait du goût et de la texture de notre produit. Nous prenons très au sérieux les préoccupations de nos clients et nous allons enquêter sur cette affaire. Pouvez-vous nous fournir plus de détails sur votre achat, notamment la date et l

New API for RunableParrallel


In [36]:
from langchain_core.runnables.base import RunnableParallel
from langchain_core.tracers.stdout import ConsoleCallbackHandler

In [37]:
first_chain = first_prompt | llm | StrOutputParser()
second_chain = second_prompt | llm | StrOutputParser()
third_chain = third_prompt | llm | StrOutputParser()
fourth_chain = fourth_prompt | llm | StrOutputParser()

overall_chain = (
    first_chain
    | {
        "summary": second_chain,
        "language": third_chain,
    }
    | fourth_chain
)
# Invoke the overall chain with the input review
result = overall_chain.invoke(
    {"Review": review},
)  # config={"callbacks": [ConsoleCallbackHandler()]})


In [38]:
print(result)

Here is a potential follow-up response in French:

"Je suis désolé d'entendre que vous avez été déçu par le goût et la texture de notre produit. Nous prenons très au sérieux les allégations de contrefaçon et nous allons enquêter immédiatement sur cette affaire. Pouvez-vous nous fournir plus de détails sur votre achat, notamment la date et le lieu d'achat, ainsi que le numéro de lot du produit ? Nous nous engageons à vous fournir un remplacement ou un remboursement si nous constatons que le produit est effectivement défectueux ou contrefait."

Translation:

"I'm sorry to hear that you were disappointed with the taste and texture of our product. We take allegations of counterfeiting very seriously and will investigate this matter immediately. Can you provide us with more details about your purchase, including the date and location of purchase, as well as the batch number of the product? We commit to providing you with a replacement or refund if we find that the product is indeed defectiv

In [39]:
print(results_old["followup_message"])

Here is a potential follow-up response in French:

"Je suis désolé d'entendre que vous n'avez pas été satisfait du goût et de la texture de notre produit. Nous prenons très au sérieux les préoccupations de nos clients et nous allons enquêter sur cette affaire. Pouvez-vous nous fournir plus de détails sur votre achat, notamment la date et le lieu d'achat, ainsi que le numéro de lot du produit ? Nous allons faire notre possible pour résoudre ce problème et vous offrir une expérience de qualité."

Translation:

"I'm sorry to hear that you were not satisfied with the taste and texture of our product. We take our customers' concerns very seriously and will investigate this matter. Can you provide us with more details about your purchase, including the date and location of purchase, as well as the batch number of the product? We will do our best to resolve this issue and offer you a quality experience."


## Router Chain(deprecated) , use combination of all 3 Runables


In [ ]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

In [ ]:
prompt_infos = [
    {
        "name": "physics",
        "description": "Good for answering questions about physics",
        "prompt_template": physics_template,
    },
    {
        "name": "math",
        "description": "Good for answering math questions",
        "prompt_template": math_template,
    },
    {
        "name": "History",
        "description": "Good for answering history questions",
        "prompt_template": history_template,
    },
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "prompt_template": computerscience_template,
    },
]

In [ ]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [ ]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.prompts import PromptTemplate

llm = ChatFireworks(temperature=0, model=llm_model)

destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations=destinations_str)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=default_chain,
    verbose=True,
)

In [ ]:
chain.run("What is black body radiation?")

In [ ]:
chain.run("what is 2 + 2? Think step by step")

In [ ]:
chain.run("Why does every cell in our body contain DNA?")

Use Combination of Runable


In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(prompt_template)
    chain = prompt | llm
    destination_chains[name] = chain

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [ ]:
destination_chains.keys()

In [ ]:
print(destinations_str)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from typing_extensions import TypedDict
from typing import Literal
from operator import itemgetter

router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations=destinations_str)
route_prompt = PromptTemplate.from_template(template=router_template)


In [ ]:
class RouteQuery(TypedDict):
    """Route query to destination."""

    destination: Literal["physics", "math", "History", "computer science"]


route_chain = (
    router_prompt | llm.with_structured_output(RouteQuery) | itemgetter("destination")
)


def condition_check(x, chain):
    return chain[x["destination"]]


chain = (
    RunnableParallel({
        "destination": route_chain,  # "animal" or "vegetable"
        "input": RunnablePassthrough(),  # pass through input query
    })
    | RunnableLambda(
        # if animal, chain_1. otherwise, chain_2.
        lambda x: condition_check(x, destination_chains)
    )
    | StrOutputParser()
)

In [ ]:
print(chain.invoke("What is 2-3*5 +2"))

In [ ]:
print(chain.invoke("What is black body radiation?"))

Try another way


In [ ]:
# Another way
from langchain.output_parsers.json import SimpleJsonOutputParser

route_chain = (
    router_prompt
    | llm  # .with_structured_output(RouteQuery)
    | SimpleJsonOutputParser()  # itemgetter("destination")
)


def condition_check(x, destination_chains):
    return destination_chains[x["classifier"]["destination"]]


chain = (
    RunnableParallel(
        {
            "classifier": route_chain,
            "input": RunnablePassthrough(),
        },  # pass through input query
    )
    | RunnableLambda(
        # if animal, chain_1. otherwise, chain_2.
        lambda x: condition_check(x, destination_chains)
    )
    | StrOutputParser()
)

In [ ]:
print(chain.invoke("What is 2-3*5 +2"))

In [ ]:
print(chain.invoke("What is black body radiation?"))